In [ ]:
try:
    import pyspark.sql.functions as F
except ModuleNotFoundError as error:
    raise RuntimeError(
        "Este notebook precisa ser executado em um cluster Databricks com PySpark."
    ) from error

SCHEMA_ORIGEM = "silver"
SCHEMA_DESTINO = "gold"
TABELA_FILMES_ORIGEM = "tb_info_filmes"
TABELA_GENEROS_ORIGEM = "tb_generos"
TABELA_PESSOAS_ORIGEM = "tb_pessoas_empresas"
TABELA_AVALIACOES_ORIGEM = "tb_avaliacoes_usuarios"

TABELA_DIM_FILMES = "dim_movies"
TABELA_DIM_GENEROS = "dim_genres"
TABELA_DIM_PESSOAS = "dim_people"
TABELA_DIM_PRODUTORAS = "dim_companies"
TABELA_DIM_AVALIACOES = "dim_reviews"
TABELA_FINANCEIRO_ORIGEM = "tb_financeiro_filmes"
TABELA_METRICAS_ORIGEM = "tb_metricas_engajamento"
TABELA_FATO_PERFORMANCE = "fact_movies_performance"
TABELA_PONTE_GENEROS = "bridge_movie_genre"
TABELA_PONTE_PESSOAS = "bridge_movie_person"
TABELA_PONTE_PRODUTORAS = "bridge_movie_company"
TABELA_CONTEXTO_GENAI = "gold_genai_movies_context"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_DESTINO}")

In [ ]:
# carrega as tabelas silver e confirma as colunas necessárias antes das dimensões
df_filmes_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_FILMES_ORIGEM}")
df_generos_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_GENEROS_ORIGEM}")
df_pessoas_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_PESSOAS_ORIGEM}")
df_avaliacoes_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_AVALIACOES_ORIGEM}")

colunas_filmes_esperadas = {
    "id_filme", "id_imdb", "titulo", "titulo_original",
    "idioma_original", "data_lancamento", "ano_lancamento",
    "duracao_minutos", "status", "sinopse", "tagline"
}
colunas_filmes_ausentes = colunas_filmes_esperadas.difference(df_filmes_silver.columns)
if colunas_filmes_ausentes:
    raise ValueError(f"Colunas ausentes na Silver de filmes: {sorted(colunas_filmes_ausentes)}")

# a chave do filme usa o id natural, mantendo o mesmo valor em cada reprocessamento
df_dim_movies = (
    df_filmes_silver
    .where(F.col("id_filme").isNotNull())
    .withColumn("sk_movie_id", F.sha2(F.concat_ws("|", F.lit("movie"), F.col("id_filme").cast("string")), 256))
    .select(
        "sk_movie_id", "id_filme", "id_imdb", "titulo",
        "titulo_original", "idioma_original", "data_lancamento",
        "ano_lancamento", "duracao_minutos", "status",
        "sinopse", "tagline"
    )
)

# grava em overwrite para permitir reprocessamento idempotente da dimensão
(df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_FILMES}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_FILMES}")
display(df_dim_movies.limit(10))

In [ ]:
# deduplica os gêneros sem diferenciar maiúsculas e minúsculas
df_dim_genres = (
    df_generos_silver
    .select(F.trim(F.col("nome_genero")).alias("nome_genero"))
    .where(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .withColumn("nome_genero_normalizado", F.lower(F.col("nome_genero")))
    .groupBy("nome_genero_normalizado")
    .agg(F.first("nome_genero", ignorenulls=True).alias("nome_genero"))
    .withColumn("sk_genre_id", F.sha2(F.concat_ws("|", F.lit("genre"), F.col("nome_genero_normalizado")), 256))
    .select("sk_genre_id", "nome_genero")
)

(df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}")
display(df_dim_genres.orderBy("nome_genero").limit(10))

In [ ]:
# separa pessoas e produtoras para que cada dimensão tenha um único tipo de entidade
df_entidades = (
    df_pessoas_silver
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_entidade"),
        F.trim(F.col("tipo_entidade")).alias("tipo_entidade")
    )
    .where(F.col("nome_entidade").isNotNull() & (F.col("nome_entidade") != ""))
)

df_dim_people = (
    df_entidades
    .where(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .dropDuplicates(["nome_entidade", "tipo_entidade"])
    .withColumn("sk_person_id", F.sha2(F.concat_ws("|", F.lit("person"), F.col("nome_entidade"), F.col("tipo_entidade")), 256))
    .select("sk_person_id", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
)

df_dim_companies = (
    df_entidades
    .where(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .dropDuplicates(["nome_produtora"])
    .withColumn("sk_company_id", F.sha2(F.concat_ws("|", F.lit("company"), F.col("nome_produtora")), 256))
    .select("sk_company_id", "nome_produtora")
)

for df_dimensao, tabela in [
    (df_dim_people, TABELA_DIM_PESSOAS),
    (df_dim_companies, TABELA_DIM_PRODUTORAS),
]:
    (df_dimensao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{SCHEMA_DESTINO}.{tabela}"
    ))
    print(f"Tabela gravada: {SCHEMA_DESTINO}.{tabela}")

display(df_dim_people.limit(10))
display(df_dim_companies.limit(10))

In [ ]:
# agrega avaliações no grão de um registro por filme
df_reviews_agregadas = (
    df_avaliacoes_silver
    .where(F.col("id_filme").isNotNull())
    .groupBy("id_filme")
    .agg(
        F.count(F.lit(1)).cast("BIGINT").alias("quantidade_avaliacoes"),
        F.round(F.avg("nota"), 2).cast("DECIMAL(10,2)").alias("nota_media")
    )
)

# relaciona a avaliação agregada com a chave substituta da dimensão de filmes
df_dim_reviews = (
    df_reviews_agregadas
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .withColumn("sk_review_id", F.sha2(F.concat_ws("|", F.lit("review"), F.col("id_filme").cast("string")), 256))
    .select("sk_review_id", "sk_movie_id", "quantidade_avaliacoes", "nota_media")
)

(df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_DIM_AVALIACOES}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_DIM_AVALIACOES}")
display(df_dim_reviews.limit(10))

In [ ]:
# carrega financeiro e engajamento para montar a fato no grão de um filme
df_financeiro_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_FINANCEIRO_ORIGEM}")
df_metricas_silver = spark.table(f"{SCHEMA_ORIGEM}.{TABELA_METRICAS_ORIGEM}")

colunas_financeiro_esperadas = {
    "id_filme", "orcamento_usd", "receita_usd",
    "cotacao_dolar_brl", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual"
}
colunas_metricas_esperadas = {
    "id_filme", "popularidade", "nota_media_tmdb",
    "quantidade_votos_tmdb", "nota_media_imdb", "quantidade_votos_imdb"
}
for colunas_esperadas, df_origem, nome_origem in [
    (colunas_financeiro_esperadas, df_financeiro_silver, "financeira"),
    (colunas_metricas_esperadas, df_metricas_silver, "métricas"),
]:
    colunas_ausentes = colunas_esperadas.difference(df_origem.columns)
    if colunas_ausentes:
        raise ValueError(f"Colunas ausentes na Silver {nome_origem}: {sorted(colunas_ausentes)}")

# joins à esquerda preservam filmes sem informação financeira ou de engajamento
df_fact_movies_performance = (
    df_dim_movies.select("id_filme", "sk_movie_id")
    .join(df_financeiro_silver, on="id_filme", how="left")
    .join(df_metricas_silver, on="id_filme", how="left")
    .select(
        "sk_movie_id",
        "orcamento_usd", "receita_usd", "cotacao_dolar_brl",
        "orcamento_brl", "receita_brl", "lucro_usd",
        "lucro_brl", "margem_lucro_percentual",
        "popularidade", "nota_media_tmdb", "quantidade_votos_tmdb",
        "nota_media_imdb", "quantidade_votos_imdb"
    )
)

(df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_FATO_PERFORMANCE}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_FATO_PERFORMANCE}")
display(df_fact_movies_performance.limit(10))

In [ ]:
# cria as tabelas-ponte a partir das relações normalizadas da silver
df_bridge_movie_genre = (
    df_generos_silver
    .select("id_filme", F.lower(F.trim(F.col("nome_genero"))).alias("nome_genero_normalizado"))
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .join(
        df_dim_genres.withColumn("nome_genero_normalizado", F.lower(F.trim(F.col("nome_genero")))),
        on="nome_genero_normalizado", how="inner"
    )
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()
)

df_bridge_movie_person = (
    df_pessoas_silver
    .where(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .join(
        df_dim_people,
        (F.trim(F.col("nome_entidade")) == F.col("nome_pessoa")) &
        (F.col("tipo_entidade") == F.col("tipo_pessoa")),
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates()
)

df_bridge_movie_company = (
    df_pessoas_silver.where(F.col("tipo_entidade") == "Produtora")
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")
    .join(df_dim_companies, F.trim(F.col("nome_entidade")) == F.col("nome_produtora"), how="inner")
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates()
)

for df_ponte, tabela in [
    (df_bridge_movie_genre, TABELA_PONTE_GENEROS),
    (df_bridge_movie_person, TABELA_PONTE_PESSOAS),
    (df_bridge_movie_company, TABELA_PONTE_PRODUTORAS),
]:
    (df_ponte.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{SCHEMA_DESTINO}.{tabela}"
    ))
    print(f"Tabela gravada: {SCHEMA_DESTINO}.{tabela}")

display(df_bridge_movie_genre.limit(10))
display(df_bridge_movie_person.limit(10))
display(df_bridge_movie_company.limit(10))

In [ ]:
# agrega os atributos relacionais e monta o texto seguro para busca semântica
df_generos_contexto = (
    df_bridge_movie_genre
    .join(df_dim_genres.select("sk_genre_id", "nome_genero"), on="sk_genre_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.sort_array(F.collect_set("nome_genero"))).alias("generos"))
)

df_pessoas_contexto = (
    df_bridge_movie_person
    .join(df_dim_people.select("sk_person_id", "nome_pessoa", "tipo_pessoa"), on="sk_person_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(
        F.concat_ws(", ", F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Ator", F.col("nome_pessoa"))))).alias("atores"),
        F.concat_ws(", ", F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Diretor", F.col("nome_pessoa"))))).alias("diretores")
    )
)

df_produtoras_contexto = (
    df_bridge_movie_company
    .join(df_dim_companies.select("sk_company_id", "nome_produtora"), on="sk_company_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.sort_array(F.collect_set("nome_produtora"))).alias("produtoras"))
)

df_metricas_contexto = df_fact_movies_performance.select("sk_movie_id", "popularidade", "nota_media_tmdb")

# coalesce e when preservam o contexto mesmo quando a origem possui nulos
def campo_contexto(nome_coluna, rotulo):
    valor = F.trim(F.coalesce(F.col(nome_coluna).cast("string"), F.lit("")))
    return F.when(valor != "", F.concat(F.lit(rotulo), valor)).otherwise(F.lit(""))

df_contexto_genai = (
    df_dim_movies.select("sk_movie_id", F.col("id_filme").alias("movie_id"), F.col("titulo").alias("title"), "sinopse")
    .join(df_generos_contexto, on="sk_movie_id", how="left")
    .join(df_pessoas_contexto, on="sk_movie_id", how="left")
    .join(df_produtoras_contexto, on="sk_movie_id", how="left")
    .join(df_metricas_contexto, on="sk_movie_id", how="left")
    .withColumn("llm_context_document", F.concat_ws(" ",
        campo_contexto("title", "Filme: "),
        campo_contexto("sinopse", "Sinopse: "),
        campo_contexto("generos", "Gêneros: "),
        campo_contexto("atores", "Atores: "),
        campo_contexto("diretores", "Diretores: "),
        campo_contexto("produtoras", "Produtoras: "),
        campo_contexto("popularidade", "Popularidade: "),
        campo_contexto("nota_media_tmdb", "Nota média TMDB: ")))
    .select("movie_id", "title", "llm_context_document")
)

(df_contexto_genai.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{SCHEMA_DESTINO}.{TABELA_CONTEXTO_GENAI}"
))

print(f"Tabela gravada: {SCHEMA_DESTINO}.{TABELA_CONTEXTO_GENAI}")
display(df_contexto_genai.limit(10))

In [ ]:
# valida chaves primárias, deduplicação e relacionamento da dimensão de avaliações
df_validacao_dim_movies = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_FILMES}")
df_validacao_dim_genres = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_GENEROS}")
df_validacao_dim_people = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_PESSOAS}")
df_validacao_dim_companies = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_PRODUTORAS}")
df_validacao_dim_reviews = spark.table(f"{SCHEMA_DESTINO}.{TABELA_DIM_AVALIACOES}")
df_validacao_contexto_genai = spark.table(f"{SCHEMA_DESTINO}.{TABELA_CONTEXTO_GENAI}")

duplicados_filmes = df_validacao_dim_movies.groupBy("sk_movie_id").count().where(F.col("count") > 1).count()
duplicados_generos = df_validacao_dim_genres.groupBy("sk_genre_id").count().where(F.col("count") > 1).count()
duplicados_pessoas = df_validacao_dim_people.groupBy("sk_person_id").count().where(F.col("count") > 1).count()
duplicados_produtoras = df_validacao_dim_companies.groupBy("sk_company_id").count().where(F.col("count") > 1).count()
duplicados_avaliacoes = df_validacao_dim_reviews.groupBy("sk_review_id").count().where(F.col("count") > 1).count()
duplicados_contexto = df_validacao_contexto_genai.groupBy("movie_id").count().where(F.col("count") > 1).count()
contexto_com_nulos = df_validacao_contexto_genai.where(
    F.col("movie_id").isNull() | F.col("title").isNull() | F.col("llm_context_document").isNull()
).count()
avaliacoes_sem_filme = (
    df_validacao_dim_reviews.join(df_validacao_dim_movies.select("sk_movie_id"), on="sk_movie_id", how="left_anti").count()
)

if any(valor != 0 for valor in [duplicados_filmes, duplicados_generos, duplicados_pessoas, duplicados_produtoras, duplicados_avaliacoes, duplicados_contexto, avaliacoes_sem_filme, contexto_com_nulos]):
    raise AssertionError(
        "As dimensões ou o contexto GenAI possuem chaves duplicadas, nulos obrigatórios ou avaliações sem filme correspondente."
    )

print(f"Filmes: {df_validacao_dim_movies.count()} | Gêneros: {df_validacao_dim_genres.count()}")
print(f"Pessoas: {df_validacao_dim_people.count()} | Produtoras: {df_validacao_dim_companies.count()}")
print(f"Filmes com avaliações agregadas: {df_validacao_dim_reviews.count()}")
print(f"Filmes com contexto GenAI: {df_validacao_contexto_genai.count()}")
print("Validação de chaves e relacionamentos concluída sem inconsistências.")